**Import all 3 Dataset**

In [ ]:
import pandas as pd

framingham = pd.read_csv("../datasets/Processed_Datasets/framingham_clean.csv")
heart = pd.read_csv("../datasets/Processed_Datasets/heart_clean.csv")
stroke_ml = pd.read_csv("../datasets/Processed_Datasets/stroke_clean_ML.csv")
stroke_dl = pd.read_csv("../datasets/Processed_Datasets/stroke_clean_DL.csv")

print("Framingham shape:", framingham.shape)
print("Framingham columns:")
print(framingham.columns.tolist())

print("\nHeart shape:", heart.shape)
print("Heart columns:")
print(heart.columns.tolist())

print("\nStroke ML shape:", stroke_ml.shape)
print("Stroke ML columns:")
print(stroke_ml.columns.tolist())

print("\nStroke DL shape:", stroke_dl.shape)
print("Stroke DL columns:")
print(stroke_dl.columns.tolist())

Framingham shape: (3658, 16)
Framingham columns:
['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'TenYearCHD']

Heart shape: (918, 12)
Heart columns:
['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease']

Stroke ML shape: (4909, 23)
Stroke ML columns:
['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'avg_glucose_level', 'bmi', 'stroke', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Urban', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes', 'risk_score', 'age_group_Adult', 'age_group_Middle_Aged', 'age_group_Elderly', 'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese']

Stroke DL shape: (4717, 16)
Stroke DL columns:
['gend

In [ ]:
print("Framingham data types:")
print(framingham.dtypes)

print("\nHeart data types:")
print(heart.dtypes)

print("\nStroke ML data types:")
print(stroke_ml.dtypes)

print("\nStroke DL data types:")
print(stroke_dl.dtypes)

Framingham data types:
male                 int64
age                  int64
education          float64
currentSmoker        int64
cigsPerDay         float64
BPMeds             float64
prevalentStroke      int64
prevalentHyp         int64
diabetes             int64
totChol            float64
sysBP              float64
diaBP              float64
BMI                float64
heartRate          float64
glucose            float64
TenYearCHD           int64
dtype: object

Heart data types:
Age                 int64
Sex                 int64
ChestPainType       int64
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG          int64
MaxHR               int64
ExerciseAngina      int64
Oldpeak           float64
ST_Slope            int64
HeartDisease        int64
dtype: object

Stroke ML data types:
gender                              int64
age                               float64
hypertension                        int64
heart_disease                       i

**in stroke_ml, some columns are bool**

In [ ]:
# Convert bool columns in stroke_ml to int
stroke_ml = stroke_ml.astype({
    col: int for col in stroke_ml.select_dtypes(include="bool").columns
})

# Framingham dataset
X_framingham = framingham.drop("TenYearCHD", axis=1)
y_framingham = framingham["TenYearCHD"]

# Heart dataset
X_heart = heart.drop("HeartDisease", axis=1)
y_heart = heart["HeartDisease"]

# Stroke ML dataset
X_stroke = stroke_ml.drop("stroke", axis=1)
y_stroke = stroke_ml["stroke"]

print("Framingham X:", X_framingham.shape, "y:", y_framingham.shape)
print("Heart X:", X_heart.shape, "y:", y_heart.shape)
print("Stroke X:", X_stroke.shape, "y:", y_stroke.shape)

Framingham X: (3658, 15) y: (3658,)
Heart X: (918, 11) y: (918,)
Stroke X: (4909, 22) y: (4909,)


**function that can train any machine learning model and return results**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

def train_ml_model(X, y, model, model_name, disease_name, scale=False):

    # Split: 60% train, 20% validation, 20% test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=0.25,
        random_state=42
    )

    # Scaling if needed
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
    else:
        scaler = None

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    # Results
    results = {
        "Disease": disease_name,
        "Model": model_name,
        "Validation Accuracy": accuracy_score(y_val, y_val_pred),
        "Validation F1 Score": f1_score(y_val, y_val_pred),
        "Test Accuracy": accuracy_score(y_test, y_test_pred),
        "Test F1 Score": f1_score(y_test, y_test_pred)
    }

    print(f"\n===== {disease_name} - {model_name} =====")
    print("Validation Accuracy:", results["Validation Accuracy"])
    print("Validation F1 Score:", results["Validation F1 Score"])
    print("Test Accuracy:", results["Test Accuracy"])
    print("Test F1 Score:", results["Test F1 Score"])
    print("\nTest Classification Report:")
    print(classification_report(y_test, y_test_pred))

    return model, scaler, results

**Training Framingham model using Logistic Regression.**

In [ ]:
from sklearn.linear_model import LogisticRegression

framingham_model = LogisticRegression(max_iter=5000,class_weight="balanced")

framingham_model, framingham_scaler, framingham_results = train_ml_model(
    X_framingham,
    y_framingham,
    framingham_model,
    "Logistic Regression",
    "Coronary Heart Disease",
    scale=True
)


===== Coronary Heart Disease - Logistic Regression =====
Validation Accuracy: 0.6762295081967213
Validation F1 Score: 0.3713527851458886
Test Accuracy: 0.6898907103825137
Test F1 Score: 0.41645244215938304

Test Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.70      0.79       610
           1       0.30      0.66      0.42       122

    accuracy                           0.69       732
   macro avg       0.61      0.68      0.60       732
weighted avg       0.81      0.69      0.73       732



**Training Heart Disease model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

heart_model = RandomForestClassifier(n_estimators=100,random_state=42)

heart_model, heart_scaler, heart_results = train_ml_model(
    X_heart,
    y_heart,
    heart_model,
    "Random Forest",
    "Heart Disease",
    scale=False
)


===== Heart Disease - Random Forest =====
Validation Accuracy: 0.907608695652174
Validation F1 Score: 0.9230769230769231
Test Accuracy: 0.8858695652173914
Test F1 Score: 0.8985507246376812

Test Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.91      0.87        77
           1       0.93      0.87      0.90       107

    accuracy                           0.89       184
   macro avg       0.88      0.89      0.88       184
weighted avg       0.89      0.89      0.89       184



**Training Stroke ML model**

In [ ]:
from sklearn.linear_model import LogisticRegression

stroke_ml_model = LogisticRegression(max_iter=5000,class_weight="balanced")

stroke_ml_model, stroke_ml_scaler, stroke_ml_results = train_ml_model(
    X_stroke,
    y_stroke,
    stroke_ml_model,
    "Logistic Regression",
    "Stroke",
    scale=True
)


===== Stroke - Logistic Regression =====
Validation Accuracy: 0.7494908350305499
Validation F1 Score: 0.2064516129032258
Test Accuracy: 0.7433808553971487
Test F1 Score: 0.25

Test Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.74      0.85       929
           1       0.15      0.79      0.25        53

    accuracy                           0.74       982
   macro avg       0.57      0.77      0.55       982
weighted avg       0.94      0.74      0.81       982



**Load Stroke DL data**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_stroke_dl = stroke_dl.drop("stroke", axis=1)
y_stroke_dl = stroke_dl["stroke"]

# Split: 60% train, 20% validation, 20% test
X_temp_dl, X_test_dl, y_temp_dl, y_test_dl = train_test_split(
    X_stroke_dl,
    y_stroke_dl,
    test_size=0.2,
    random_state=42
)

X_train_dl, X_val_dl, y_train_dl, y_val_dl = train_test_split(
    X_temp_dl,
    y_temp_dl,
    test_size=0.25,
    random_state=42
)

# Scale for ANN
dl_scaler = StandardScaler()

X_train_dl = dl_scaler.fit_transform(X_train_dl)
X_val_dl = dl_scaler.transform(X_val_dl)
X_test_dl = dl_scaler.transform(X_test_dl)

print("X_train_dl:", X_train_dl.shape)
print("X_val_dl:", X_val_dl.shape)
print("X_test_dl:", X_test_dl.shape)

print("y_train_dl:", y_train_dl.shape)
print("y_val_dl:", y_val_dl.shape)
print("y_test_dl:", y_test_dl.shape)

X_train_dl: (2829, 15)
X_val_dl: (944, 15)
X_test_dl: (944, 15)
y_train_dl: (2829,)
y_val_dl: (944,)
y_test_dl: (944,)


**Train Stroke ANN model**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np

# Build ANN model
stroke_ann = Sequential()

stroke_ann.add(Input(shape=(X_train_dl.shape[1],)))

stroke_ann.add(Dense(16, activation='relu'))
stroke_ann.add(Dropout(0.2))

stroke_ann.add(Dense(8, activation='relu'))
stroke_ann.add(Dropout(0.2))

stroke_ann.add(Dense(1, activation='sigmoid'))

# Compile model
stroke_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

stroke_ann.summary()

# Class weights for imbalance
classes = np.unique(y_train_dl)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train_dl
)

class_weights = dict(zip(classes, class_weights))

print("Class weights:", class_weights)

# Train model
stroke_history = stroke_ann.fit(
    X_train_dl,
    y_train_dl,
    validation_data=(X_val_dl, y_val_dl),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    verbose=1
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 16)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401 (1.57 KB)

 Trainable params: 401 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

Class weights: {0.0: 0.5221483942414175, 1.0: 11.7875}
Epoch 1/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8791 - loss: 0.7239 - val_accuracy: 0.8686 - val_loss: 0.5148
Epoch 2/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8021 - loss: 0.6763 - val_accuracy: 0.7860 - val_loss: 0.5220
Epoch 3/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7515 - loss: 0.6357 - val_accuracy: 0.7383 - val_loss: 0.5117
Epoch 4/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7215 - loss: 0.6252 - val_accuracy: 0.7013 - val_loss: 0.4964
Epoch 5/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6964 - loss: 0.6074 - val_accuracy: 0.6811 - val_loss: 0.4852
Epoch 6/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6713 - loss: 0.5907 - val_accuracy: 0.6653 - val_loss: 0.4859
Epoch 7/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6695 - loss: 0.5634 - val_accuracy: 0.6536 - val_loss: 0.4836
Epoch 8/50
89/89 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0

In [ ]:
# Predict probabilities
y_val_prob_dl = stroke_ann.predict(X_val_dl)
y_test_prob_dl = stroke_ann.predict(X_test_dl)

# Convert probabilities to classes
y_val_pred_dl = (y_val_prob_dl >= 0.5).astype(int)
y_test_pred_dl = (y_test_prob_dl >= 0.5).astype(int)

# Results
stroke_dl_results = {
    "Disease": "Stroke",
    "Model": "Deep Learning ANN",
    "Validation Accuracy": accuracy_score(y_val_dl, y_val_pred_dl),
    "Validation F1 Score": f1_score(y_val_dl, y_val_pred_dl),
    "Test Accuracy": accuracy_score(y_test_dl, y_test_pred_dl),
    "Test F1 Score": f1_score(y_test_dl, y_test_pred_dl)
}

print(stroke_dl_results)

print("\nTest Classification Report:")
print(classification_report(y_test_dl, y_test_pred_dl))

30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
{'Disease': 'Stroke', 'Model': 'Deep Learning ANN', 'Validation Accuracy': 0.763771186440678, 'Validation F1 Score': 0.17712177121771217, 'Test Accuracy': 0.7860169491525424, 'Test F1 Score': 0.3082191780821918}

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      0.78      0.87       889
         1.0       0.19      0.82      0.31        55

    accuracy                           0.79       944
   macro avg       0.59      0.80      0.59       944
weighted avg       0.94      0.79      0.84       944



**Deep Learning ANN for CHD**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

In [ ]:
# CHD / Framingham data
X_chd_dl = framingham.drop("TenYearCHD", axis=1)
y_chd_dl = framingham["TenYearCHD"]

# Split: 60% train, 20% validation, 20% test
X_temp_chd, X_test_chd, y_temp_chd, y_test_chd = train_test_split(
    X_chd_dl,
    y_chd_dl,
    test_size=0.2,
    random_state=42
)

X_train_chd, X_val_chd, y_train_chd, y_val_chd = train_test_split(
    X_temp_chd,
    y_temp_chd,
    test_size=0.25,
    random_state=42
)

# Scaling
chd_dl_scaler = StandardScaler()

X_train_chd = chd_dl_scaler.fit_transform(X_train_chd)
X_val_chd = chd_dl_scaler.transform(X_val_chd)
X_test_chd = chd_dl_scaler.transform(X_test_chd)

print("X_train_chd:", X_train_chd.shape)
print("X_val_chd:", X_val_chd.shape)
print("X_test_chd:", X_test_chd.shape)

X_train_chd: (2194, 15)
X_val_chd: (732, 15)
X_test_chd: (732, 15)


In [ ]:
chd_ann = Sequential()

chd_ann.add(Input(shape=(X_train_chd.shape[1],)))

chd_ann.add(Dense(16, activation='relu'))
chd_ann.add(Dropout(0.2))

chd_ann.add(Dense(8, activation='relu'))
chd_ann.add(Dropout(0.2))

chd_ann.add(Dense(1, activation='sigmoid'))

chd_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

chd_ann.summary()

# Class weights
classes_chd = np.unique(y_train_chd)

class_weights_chd = compute_class_weight(
    class_weight='balanced',
    classes=classes_chd,
    y=y_train_chd
)

class_weights_chd = dict(zip(classes_chd, class_weights_chd))

print("CHD class weights:", class_weights_chd)

# Train model
chd_history = chd_ann.fit(
    X_train_chd,
    y_train_chd,
    validation_data=(X_val_chd, y_val_chd),
    epochs=50,
    batch_size=32,
    class_weight=class_weights_chd,
    verbose=1
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 16)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 401 (1.57 KB)

 Trainable params: 401 (1.57 KB)

 Non-trainable params: 0 (0.00 B)

CHD class weights: {0: 0.5904198062432723, 1: 3.2648809523809526}
Epoch 1/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2967 - loss: 0.7505 - val_accuracy: 0.3784 - val_loss: 0.7664
Epoch 2/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4531 - loss: 0.6881 - val_accuracy: 0.5232 - val_loss: 0.7139
Epoch 3/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5611 - loss: 0.6762 - val_accuracy: 0.5929 - val_loss: 0.6859
Epoch 4/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6098 - loss: 0.6593 - val_accuracy: 0.6544 - val_loss: 0.6560
Epoch 5/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6208 - loss: 0.6546 - val_accuracy: 0.6462 - val_loss: 0.6618
Epoch 6/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6381 - loss: 0.6450 - val_accuracy: 0.6530 - val_loss: 0.6530
Epoch 7/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6504 - loss: 0.6499 - val_accuracy: 0.6585 - val_loss: 0.6493
Epoch 8/50
69/69 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - 

In [ ]:
# Predict probabilities
y_val_prob_chd = chd_ann.predict(X_val_chd)
y_test_prob_chd = chd_ann.predict(X_test_chd)


y_val_pred_chd = (y_val_prob_chd >= 0.3).astype(int)
y_test_pred_chd = (y_test_prob_chd >= 0.3).astype(int)

chd_dl_results = {
    "Disease": "Coronary Heart Disease",
    "Model": "Deep Learning ANN",
    "Validation Accuracy": accuracy_score(y_val_chd, y_val_pred_chd),
    "Validation F1 Score": f1_score(y_val_chd, y_val_pred_chd),
    "Test Accuracy": accuracy_score(y_test_chd, y_test_pred_chd),
    "Test F1 Score": f1_score(y_test_chd, y_test_pred_chd)
}

print(chd_dl_results)

print("\nCHD Test Classification Report:")
print(classification_report(y_test_chd, y_test_pred_chd))

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
{'Disease': 'Coronary Heart Disease', 'Model': 'Deep Learning ANN', 'Validation Accuracy': 0.4057377049180328, 'Validation F1 Score': 0.2903752039151713, 'Test Accuracy': 0.4426229508196721, 'Test F1 Score': 0.3419354838709677}

CHD Test Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.36      0.52       610
           1       0.21      0.87      0.34       122

    accuracy                           0.44       732
   macro avg       0.57      0.61      0.43       732
weighted avg       0.81      0.44      0.49       732



**Deep Learning ANN for Heart Disease**

In [ ]:
# Heart Disease data
X_heart_dl = heart.drop("HeartDisease", axis=1)
y_heart_dl = heart["HeartDisease"]

# Split: 60% train, 20% validation, 20% test
X_temp_heart, X_test_heart, y_temp_heart, y_test_heart = train_test_split(
    X_heart_dl,
    y_heart_dl,
    test_size=0.2,
    random_state=42
)

X_train_heart, X_val_heart, y_train_heart, y_val_heart = train_test_split(
    X_temp_heart,
    y_temp_heart,
    test_size=0.25,
    random_state=42
)

# Scaling
heart_dl_scaler = StandardScaler()

X_train_heart = heart_dl_scaler.fit_transform(X_train_heart)
X_val_heart = heart_dl_scaler.transform(X_val_heart)
X_test_heart = heart_dl_scaler.transform(X_test_heart)

print("X_train_heart:", X_train_heart.shape)
print("X_val_heart:", X_val_heart.shape)
print("X_test_heart:", X_test_heart.shape)

X_train_heart: (550, 11)
X_val_heart: (184, 11)
X_test_heart: (184, 11)


In [ ]:
heart_ann = Sequential()

heart_ann.add(Input(shape=(X_train_heart.shape[1],)))

heart_ann.add(Dense(16, activation='relu'))
heart_ann.add(Dropout(0.2))

heart_ann.add(Dense(8, activation='relu'))
heart_ann.add(Dropout(0.2))

heart_ann.add(Dense(1, activation='sigmoid'))

heart_ann.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

heart_ann.summary()

# Class weights
classes_heart = np.unique(y_train_heart)

class_weights_heart = compute_class_weight(
    class_weight='balanced',
    classes=classes_heart,
    y=y_train_heart
)

class_weights_heart = dict(zip(classes_heart, class_weights_heart))

print("Heart class weights:", class_weights_heart)

# Train model
heart_history = heart_ann.fit(
    X_train_heart,
    y_train_heart,
    validation_data=(X_val_heart, y_val_heart),
    epochs=50,
    batch_size=32,
    class_weight=class_weights_heart,
    verbose=1
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 16)             │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 337 (1.32 KB)

 Trainable params: 337 (1.32 KB)

 Non-trainable params: 0 (0.00 B)

Heart class weights: {0: 1.0784313725490196, 1: 0.9322033898305084}
Epoch 1/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5855 - loss: 0.6512 - val_accuracy: 0.6902 - val_loss: 0.5691
Epoch 2/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6473 - loss: 0.6239 - val_accuracy: 0.7772 - val_loss: 0.5331
Epoch 3/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6873 - loss: 0.5848 - val_accuracy: 0.8370 - val_loss: 0.5047
Epoch 4/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7200 - loss: 0.5635 - val_accuracy: 0.8641 - val_loss: 0.4774
Epoch 5/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7455 - loss: 0.5533 - val_accuracy: 0.8641 - val_loss: 0.4566
Epoch 6/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7618 - loss: 0.5283 - val_accuracy: 0.8750 - val_loss: 0.4373
Epoch 7/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7764 - loss: 0.5156 - val_accuracy: 0.8750 - val_loss: 0.4169
Epoch 8/50
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step

In [ ]:
# Predict probabilitie
y_val_prob_heart = heart_ann.predict(X_val_heart)
y_test_prob_heart = heart_ann.predict(X_test_heart)


y_val_pred_heart = (y_val_prob_heart >= 0.5).astype(int)
y_test_pred_heart = (y_test_prob_heart >= 0.5).astype(int)

heart_dl_results = {
    "Disease": "Heart Disease",
    "Model": "Deep Learning ANN",
    "Validation Accuracy": accuracy_score(y_val_heart, y_val_pred_heart),
    "Validation F1 Score": f1_score(y_val_heart, y_val_pred_heart),
    "Test Accuracy": accuracy_score(y_test_heart, y_test_pred_heart),
    "Test F1 Score": f1_score(y_test_heart, y_test_pred_heart)
}

print(heart_dl_results)

print("\nHeart Test Classification Report:")
print(classification_report(y_test_heart, y_test_pred_heart))

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
{'Disease': 'Heart Disease', 'Model': 'Deep Learning ANN', 'Validation Accuracy': 0.8967391304347826, 'Validation F1 Score': 0.9140271493212669, 'Test Accuracy': 0.8641304347826086, 'Test F1 Score': 0.8803827751196173}

Heart Test Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84        77
           1       0.90      0.86      0.88       107

    accuracy                           0.86       184
   macro avg       0.86      0.86      0.86       184
weighted avg       0.87      0.86      0.86       184



**Finals Result ML + DL**

In [ ]:
final_results = pd.DataFrame([
    framingham_results,
    chd_dl_results,
    heart_results,
    heart_dl_results,
    stroke_ml_results,
    stroke_dl_results
])

final_results

,Disease,Model,Validation Accuracy,Validation F1 Score,Test Accuracy,Test F1 Score
0,Coronary Heart Disease,Logistic Regression,0.676230,0.371353,0.689891,0.416452
1,Coronary Heart Disease,Deep Learning ANN,0.405738,0.290375,0.442623,0.341935
2,Heart Disease,Random Forest,0.907609,0.923077,0.885870,0.898551
3,Heart Disease,Deep Learning ANN,0.896739,0.914027,0.864130,0.880383
4,Stroke,Logistic Regression,0.749491,0.206452,0.743381,0.250000
5,Stroke,Deep Learning ANN,0.763771,0.177122,0.786017,0.308219


**Best results for each model**

In [ ]:
best_final_results = pd.DataFrame([
    framingham_results,
    heart_results,
    stroke_dl_results
])

best_final_results

,Disease,Model,Validation Accuracy,Validation F1 Score,Test Accuracy,Test F1 Score
0,Coronary Heart Disease,Logistic Regression,0.676230,0.371353,0.689891,0.416452
1,Heart Disease,Random Forest,0.907609,0.923077,0.885870,0.898551
2,Stroke,Deep Learning ANN,0.763771,0.177122,0.786017,0.308219


**Final prediction system**

In [ ]:
def final_prediction_system(chd_input, heart_input, stroke_input):

    # 1. CHD prediction using Logistic Regression
    chd_input_scaled = framingham_scaler.transform(chd_input)
    chd_prediction = framingham_model.predict(chd_input_scaled)
    chd_probability = framingham_model.predict_proba(chd_input_scaled)[:, 1]

    # 2. Heart Disease prediction using Random Forest
    heart_prediction = heart_model.predict(heart_input)
    heart_probability = heart_model.predict_proba(heart_input)[:, 1]

    # 3. Stroke prediction using Logistic Regression
    stroke_input_scaled = stroke_ml_scaler.transform(stroke_input)
    stroke_prediction = stroke_ml_model.predict(stroke_input_scaled)
    stroke_probability = stroke_ml_model.predict_proba(stroke_input_scaled)[:, 1]

    results = {
        "Coronary Heart Disease Prediction": int(chd_prediction[0]),
        "Coronary Heart Disease Probability": float(chd_probability[0]),

        "Heart Disease Prediction": int(heart_prediction[0]),
        "Heart Disease Probability": float(heart_probability[0]),

        "Stroke Prediction": int(stroke_prediction[0]),
        "Stroke Probability": float(stroke_probability[0])
    }

    return results

In [ ]:
def display_final_predictions(results):
    for disease, prediction_key, probability_key in [
        ("Coronary Heart Disease",
         "Coronary Heart Disease Prediction",
         "Coronary Heart Disease Probability"),

        ("Heart Disease",
         "Heart Disease Prediction",
         "Heart Disease Probability"),

        ("Stroke",
         "Stroke Prediction",
         "Stroke Probability")
    ]:
        prediction = results[prediction_key]
        probability = results[probability_key]

        if prediction == 1:
            status = "Disease predicted"
        else:
            status = "Disease not predicted"

        print(f"{disease}: {status}")
        print(f"Risk probability: {probability:.2f}")
        print("-" * 40)

**Random testing patient**

In [ ]:
random_chd_patient = pd.DataFrame([{
    "male": 1,
    "age": 62,
    "education": 2,
    "currentSmoker": 1,
    "cigsPerDay": 15,
    "BPMeds": 1,
    "prevalentStroke": 0,
    "prevalentHyp": 1,
    "diabetes": 1,
    "totChol": 260,
    "sysBP": 155,
    "diaBP": 95,
    "BMI": 31.0,
    "heartRate": 85,
    "glucose": 140
}])

random_heart_patient = pd.DataFrame([{
    "Age": 62,
    "Sex": 1,
    "ChestPainType": 2,
    "RestingBP": 155,
    "Cholesterol": 260,
    "FastingBS": 1,
    "RestingECG": 1,
    "MaxHR": 120,
    "ExerciseAngina": 1,
    "Oldpeak": 2.0,
    "ST_Slope": 1
}])

random_stroke_patient = pd.DataFrame([{
    "gender": 1,
    "age": 62,
    "hypertension": 1,
    "heart_disease": 1,
    "ever_married": 1,
    "avg_glucose_level": 140,
    "bmi": 31.0,
    "work_type_Never_worked": 0,
    "work_type_Private": 1,
    "work_type_Self-employed": 0,
    "work_type_children": 0,
    "Residence_type_Urban": 1,
    "smoking_status_formerly smoked": 0,
    "smoking_status_never smoked": 0,
    "smoking_status_smokes": 1,
    "risk_score": 5,
    "age_group_Adult": 0,
    "age_group_Middle_Aged": 0,
    "age_group_Elderly": 1,
    "bmi_category_Normal": 0,
    "bmi_category_Overweight": 0,
    "bmi_category_Obese": 1
}])

In [ ]:
random_results = final_prediction_system(
    random_chd_patient,
    random_heart_patient,
    random_stroke_patient
)

display_final_predictions(random_results)

Coronary Heart Disease: Disease predicted
Risk probability: 0.92
----------------------------------------
Heart Disease: Disease predicted
Risk probability: 0.83
----------------------------------------
Stroke: Disease predicted
Risk probability: 0.52
----------------------------------------


**Another patient to test**

In [ ]:
random_chd_patient_2 = pd.DataFrame([{
    "male": 0,
    "age": 48,
    "education": 3,
    "currentSmoker": 0,
    "cigsPerDay": 0,
    "BPMeds": 0,
    "prevalentStroke": 0,
    "prevalentHyp": 1,
    "diabetes": 0,
    "totChol": 220,
    "sysBP": 138,
    "diaBP": 86,
    "BMI": 27.0,
    "heartRate": 76,
    "glucose": 100
}])

random_heart_patient_2 = pd.DataFrame([{
    "Age": 48,
    "Sex": 0,
    "ChestPainType": 2,
    "RestingBP": 138,
    "Cholesterol": 220,
    "FastingBS": 0,
    "RestingECG": 1,
    "MaxHR": 150,
    "ExerciseAngina": 0,
    "Oldpeak": 0.8,
    "ST_Slope": 1
}])

random_stroke_patient_2 = pd.DataFrame([{
    "gender": 0,
    "age": 48,
    "hypertension": 1,
    "heart_disease": 0,
    "ever_married": 1,
    "avg_glucose_level": 100,
    "bmi": 27.0,
    "work_type_Never_worked": 0,
    "work_type_Private": 1,
    "work_type_Self-employed": 0,
    "work_type_children": 0,
    "Residence_type_Urban": 1,
    "smoking_status_formerly smoked": 0,
    "smoking_status_never smoked": 1,
    "smoking_status_smokes": 0,
    "risk_score": 2,
    "age_group_Adult": 0,
    "age_group_Middle_Aged": 1,
    "age_group_Elderly": 0,
    "bmi_category_Normal": 0,
    "bmi_category_Overweight": 1,
    "bmi_category_Obese": 0
}])

In [ ]:
random_results_2 = final_prediction_system(
    random_chd_patient_2,
    random_heart_patient_2,
    random_stroke_patient_2
)

display_final_predictions(random_results_2)

Coronary Heart Disease: Disease not predicted
Risk probability: 0.30
----------------------------------------
Heart Disease: Disease not predicted
Risk probability: 0.23
----------------------------------------
Stroke: Disease not predicted
Risk probability: 0.16
----------------------------------------


**Another sample of a patient**

In [ ]:
random_chd_patient_3 = pd.DataFrame([{
    "male": 1,
    "age": 40,
    "education": 3,
    "currentSmoker": 0,
    "cigsPerDay": 0,
    "BPMeds": 0,
    "prevalentStroke": 0,
    "prevalentHyp": 0,
    "diabetes": 0,
    "totChol": 190,
    "sysBP": 120,
    "diaBP": 78,
    "BMI": 24.5,
    "heartRate": 72,
    "glucose": 88
}])

random_heart_patient_3 = pd.DataFrame([{
    "Age": 40,
    "Sex": 1,
    "ChestPainType": 2,
    "RestingBP": 120,
    "Cholesterol": 190,
    "FastingBS": 0,
    "RestingECG": 1,
    "MaxHR": 170,
    "ExerciseAngina": 0,
    "Oldpeak": 0.2,
    "ST_Slope": 1
}])

random_stroke_patient_3 = pd.DataFrame([{
    "gender": 1,
    "age": 40,
    "hypertension": 0,
    "heart_disease": 0,
    "ever_married": 1,
    "avg_glucose_level": 88,
    "bmi": 24.5,
    "work_type_Never_worked": 0,
    "work_type_Private": 1,
    "work_type_Self-employed": 0,
    "work_type_children": 0,
    "Residence_type_Urban": 1,
    "smoking_status_formerly smoked": 0,
    "smoking_status_never smoked": 1,
    "smoking_status_smokes": 0,
    "risk_score": 1,
    "age_group_Adult": 1,
    "age_group_Middle_Aged": 0,
    "age_group_Elderly": 0,
    "bmi_category_Normal": 1,
    "bmi_category_Overweight": 0,
    "bmi_category_Obese": 0
}])

In [ ]:
random_results_3 = final_prediction_system(
    random_chd_patient_3,
    random_heart_patient_3,
    random_stroke_patient_3
)

display_final_predictions(random_results_3)

Coronary Heart Disease: Disease not predicted
Risk probability: 0.23
----------------------------------------
Heart Disease: Disease not predicted
Risk probability: 0.37
----------------------------------------
Stroke: Disease not predicted
Risk probability: 0.05
----------------------------------------
